# Vector stores and semantic search



The initial `ScentenceTransformer` will be `all-MiniLM-L6-v2` as seen in class.

In [1]:
from sentence_transformers import SentenceTransformer, util
import numpy as np
import pandas as pd
import torch

model = SentenceTransformer("all-MiniLM-L6-v2") # 384 dimensions

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Then, we can make our own `Document`, `SearchResult` and `VectorStore` classes.

`Document` and `SearchResult` are plain model classes, whereas `VectorStore` will allow us to store documents and query them using cosine similarity.

## Part I: Basic vector store implementation

In [2]:
class Document:
    def __init__(self, text: str, metadata: dict[str, str]):
        self.text = text
        self.metadata = metadata


class SearchResult:
    def __init__(self, score: float, document: Document):
        self.score = score
        self.document = document


class VectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.embedding_model = embedding_model
        self.documents = []
        self.embeddings = None

    def add_documents(self, documents: list[Document]):
        # add all documents
        self.documents.extend(documents)

        # get their content
        texts = [doc.text for doc in documents]

        # get their embeddings
        new_embeddings = self.embedding_model.encode(texts)

        # replace new embeddings with old ones or add them
        if self.embeddings is None:
            self.embeddings = new_embeddings
        else:
            self.embeddings = np.vstack([self.embeddings, new_embeddings])

    def search(self, query: str, top_k: int = 5) -> list[SearchResult]:
        # get the query embedding
        query_embedding = self.embedding_model.encode(query)

        # compute cosine similarity scores between the query embedding and all document embeddings
        # according to docuemntation, [0] gives the per-document score vector
        scores = util.cos_sim(query_embedding, self.embeddings)[0]
        
        # get the top k scores and their indices
        top = torch.topk(scores, k=min(top_k, len(self.documents)))
        
        # return a search result for each top score and its corresponding document
        return [SearchResult(float(score), self.documents[i]) for score, i in zip(top.values, top.indices)]

As seen in class, we used `SentenceTransformer.util.cos_sim` to get the cosine similarity. According to its documentation, only the first row of the array `[0]` is useful to us, because it's the documentation scores. https://docs.pytorch.org/docs/2.12/generated/torch.topk.html

Additionally, we use `torch.topk` (https://docs.pytorch.org/docs/2.12/generated/torch.topk.html) to use those scores and select the top documents. I could have done this in a more manual way but this is more concise and easier to read. 

Now, we can read our data and start adding the documents pertaining to it.

Particularily, we will use the [Animal Fun Facts Dataset](https://github.com/ekohrt/animal-fun-facts-dataset). We will impute missing values with empty strings. The process is as follows:

1. Read the data
2. Iterate through all rows
3. For each row, create a metadata dictionary with `animal_name`, `source`, `media_link`, `wikipedia_link`.
4. For each row, create a document with its text (`row["text"]`) and its created metadata.
5. Create our `VectorStore` (class we created) based on our selected model.
6. Add all of our created documents

In [3]:
# load
df = pd.read_csv("data/animal-fun-facts-dataset.csv")
df = df.fillna("") # replace NaN with empty string

# iterate through all rows
# 1. create metadata
# 2. create a document with the text and metadta
animal_documents = []
for _, row in df.iterrows():
    metadata = {
        "animal_name": row["animal_name"],
        "source": row["source"],
        "media_link": row["media_link"],
        "wikipedia_link": row["wikipedia_link"],
    }
    animal_documents.append(Document(row["text"], metadata))

store = VectorStore(model)
store.add_documents(animal_documents)

Let's see how many documents are in

In [4]:
print(f"Documents in store: {len(store.documents)}")

Documents in store: 7734


Now, our vector store is created and we can query stuff. We just create our query array and call `store.search on all of them`. We will make 5 queries.

_Disclaimer: I've generated the queries with ChatGPT for simplicity._

In [5]:
# quieres generated with ChatGPt
queries = [
    "Which animal has the highest hunting success rate?",
    "Animals that can change their color",
    "What is the fastest land animal?",
    "Animals with strange or unusual teeth",
    "Which animals are dangerous to humans?",
]

for query in queries:
    print(f"Query: {query}")

    results = store.search(query, top_k=3)
    for result in results:
        print(f"\t[{result.score:.3f}] {result.document.text}")
        print(f"\t\tmetadata: {result.document.metadata}")
        
    print()

Query: Which animal has the highest hunting success rate?
	[0.733] They are the most efficient hunters of any large predator with an 80% success rate.
		metadata: {'animal_name': 'african wild dog', 'source': 'https://www.animalfactsencyclopedia.com/African-wild-dog-facts.html', 'media_link': '', 'wikipedia_link': '/wiki/African_wild_dog'}
	[0.616] They're one of the oldest hunting breeds on Earth.
		metadata: {'animal_name': 'spinone italiano', 'source': 'https://a-z-animals.com/animals/spinone-italiano/', 'media_link': '', 'wikipedia_link': '/wiki/Spinone_Italiano'}
	[0.594] Servals are successful in 50 percent of their hunts
		metadata: {'animal_name': 'serval cat', 'source': 'https://www.animalfactsencyclopedia.com/Serval-cat.html', 'media_link': '', 'wikipedia_link': '/wiki/Serval'}

Query: Animals that can change their color
	[0.737] Some species can change color from dark to light, and back again.
		metadata: {'animal_name': 'dwarf boa', 'source': 'https://a-z-animals.com/animal

## Part II: Filtering by metadata

To make a `FilteredVectorStore`, all we need to do is change the `search` function. We add a `metadata_filter` param, a dictionary that maps a key-value. 

In `search` we if the filter is `None` we can behave as usual. If it's present, we can manually iterate through all documents and select those whose metadata matches EXACTLY what the filter specifies. Each key-value pair has to match, if one fails, then the document is discarded. 

To simplify selection, we first order the documents by score because we can then assume that each hit we get on a document is the next in the top K list.

In [6]:
class FilteredVectorStore:
    def __init__(self, embedding_model: SentenceTransformer):
        self.embedding_model = embedding_model
        self.documents: list[Document] = []
        self.embeddings = None

    def add_documents(self, documents: list[Document]):
        self.documents.extend(documents)
        texts = [doc.text for doc in documents]
        new_embeddings = self.embedding_model.encode(texts)
        if self.embeddings is None:
            self.embeddings = new_embeddings
        else:
            self.embeddings = np.vstack([self.embeddings, new_embeddings])

    def search(self,
               query: str,
               top_k: int = 5,
               metadata_filter: dict[str, str] | None = None) -> list[SearchResult]:
        # get the query embedding
        query_embedding = self.embedding_model.encode(query)

        # compute cosine similarity scores between the query embedding and all document embeddings
        # according to docuemntation, [0] gives the per-document score vector
        scores = util.cos_sim(query_embedding, self.embeddings)[0]

        if metadata_filter is None:
            # if no filter is provided, just return the top K results as usual
            # same code as VectorStore
            top = torch.topk(scores, k=min(top_k, len(self.documents)))
            return [SearchResult(float(score), self.documents[i]) for score, i in zip(top.values, top.indices)]

        # order them by score because we're gonna iterate 1 by 1 and select until we have K
        # matches on the filter
        order = torch.argsort(scores, descending=True)

        # iterate
        results = []
        for i in order:
            # get the doc
            doc = self.documents[i]

            # compare each key-value in the filter with the document's metadata
            # if they ALL() match then we can assume the document matches and we add uit
            if all(doc.metadata.get(k) == v for k, v in metadata_filter.items()):
                results.append(SearchResult(float(scores[i]), doc))
            
            # if we've got top K then we can return
            if len(results) == top_k:
                break

        return results


Next, we can use our filtered vector store. This time, we will use [News Category Dataset](https://www.kaggle.com/datasets/rmisra/news-category-dataset). It's got ~210,000 entries, of which we will select only a portion.

Same as before, we:
1. Read data
2. Per row, create the metadata
3. Per row, save the document
4. Create our `FilteredVectorStore`
5. And, add the documents.

_Note: we limit this data to 3000 rows because the dataset is pretty huge._

In [7]:
news_df = pd.read_json("data/News_Category_Dataset_v3.json", lines=True)
news_df = news_df.sample(3000, random_state=42)

news_documents = []
for _, row in news_df.iterrows():
    metadata = {
        "category": row["category"],
        "authors": row["authors"],
        "date": str(row["date"].date()),
        "link": row["link"],
    }
    news_documents.append(Document(row["headline"], metadata))

filtered_store = FilteredVectorStore(model)
filtered_store.add_documents(news_documents)

Print the length

In [8]:
print(f"Documents in store: {len(filtered_store.documents)}")

Documents in store: 3000


Then, let's make the queries.

_Disclaimer: once again, the queries are made with ChatGPT. This time, I've added the category filter on each one._

In [9]:
# quieres generated with ChatGPt
filtered_queries = [
    ("election campaign results", {"category": "POLITICS"}),
    ("new movie premiere", {"category": "ENTERTAINMENT"}),
    ("tips for a healthy life", {"category": "WELLNESS"}),
    ("championship basketball game", {"category": "SPORTS"}),
    ("stock market and economy", {"category": "BUSINESS"}),
]

for query, metadata_filter in filtered_queries:
    print(f"Query: {query}")
    print(f"Filter: {metadata_filter}")

    results = filtered_store.search(query, top_k=3, metadata_filter=metadata_filter)
    for result in results:
        print(f"\t[{result.score:.3f}] {result.document.text}")
        print(f"\t\tmetadata: {result.document.metadata}")

    print()

Query: election campaign results
Filter: {'category': 'POLITICS'}
	[0.662] My Candidates Lost: Now What?
		metadata: {'category': 'POLITICS', 'authors': 'Amy Arndt, ContributorHumorist, Blogger, The Amy Situation', 'date': '2014-11-29', 'link': 'https://www.huffingtonpost.com/entry/my-candidates-lost-now-what_b_6240814.html'}
	[0.488] The Victory Of Wall Street Democrats
		metadata: {'category': 'POLITICS', 'authors': '', 'date': '2014-07-27', 'link': 'https://www.huffingtonpost.com/entry/the-victory-of-wall-stree_n_5624883.html'}
	[0.464] HUFFPOLLSTER: How Hillary Clinton Could Win In A Landslide
		metadata: {'category': 'POLITICS', 'authors': 'Ariel Edwards-Levy and Janie Velencia', 'date': '2016-08-15', 'link': 'https://www.huffingtonpost.com/entry/hillary-clinton-could-win-in-a-landslide_us_57b1ae13e4b071840411d76c'}

Query: new movie premiere
Filter: {'category': 'ENTERTAINMENT'}
	[0.392] New ‘Incredibles 2’ Trailer Is All About Mom’s New Job And Dad Staying At Home
		metadata: {'

## Reflexiones personales

All in all, it is important to understand the use of embeddings and vector stores in the world of Natural Language Processing. Without embeddings, it would be very hard to establish relationships between text fragments and have a numeric representation of texts that can be used to compute calculations such as queries and filters. In this case, vector stores are what are used to store embeddings and represent a collection of documents that we can act upon. The point of this exercise was not to only understand how embeddings and vector stores work and how they fit in with each other, but also to learn how to use them and take advantage of them for our own real world applications.

Particularily, I can make the following conclusions out of this exercise:
1. Embeddings are imperative because they are the foundation of what vector stores work on. The numeric representation of text allows us to compute calculations such as cosine similarity for search.
2. Vector stores are just like any other collection. Just, in this case, they store embeddings that we can act upon.
3. Metadata is very important. They can allow us to do filters and reduce the dimension of our search to a lower time complexity. They also provide very insighftul information on our data,


In general, this exercise helped me learn a lot on vector stores and what they do, how they work, and how I can use them to solve real world problems. I was amazed by:
1. How fast they are (no benchmarks made, I'm just eyeballing it)
2. How accurate they are (I assume, because of embeddings)